In [2]:
# random.ipynb
"""
在 random.ipynb 里直接运行：
1) 先加载 greedy_neuralUCB.ipynb 的基础代码（除最后示例运行部分）
2) 仅将路由策略改为随机选择
3) 复用原训练/评估/记录逻辑
"""

import json
import random
from pathlib import Path


def _load_greedy_core(notebook_path: str = "./greedy_neuralUCB.ipynb") -> None:
    nb_path = Path(notebook_path)
    if not nb_path.exists():
        # 兼容从项目根目录启动的情况
        alt = Path.cwd() / "Algorithm" / "greedy_neuralUCB.ipynb"
        if alt.exists():
            nb_path = alt
        else:
            raise FileNotFoundError(f"找不到 notebook: {notebook_path}")

    with open(nb_path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    loaded_cells = 0
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue

        src = "\n".join(cell.get("source", []))
        if not src.strip():
            continue

        # 去掉 greedy 笔记本末尾的自动训练示例，避免一加载就长时间训练
        if "report = train_with_controls(" in src:
            marker = "#===== 使用示例（按需修改） ====="
            if marker in src:
                src = src.split(marker, 1)[0]
            else:
                continue

        exec(src, globals())
        loaded_cells += 1

    print(f"✅ 已加载 greedy 核心代码，共 {loaded_cells} 个代码单元")


_load_greedy_core("./greedy_neuralUCB.ipynb")


class RandomRoutingNeuralUCBModel(GreedyNeuralUCBModel):
    def __init__(self, *args, seed=42, **kwargs):
        super().__init__(*args, seed=seed, **kwargs)
        self._router_rng = random.Random(seed)

    def select_model(self, feature_vector):
        """随机路由：忽略 UCB，均匀随机选模型。"""
        if not self.model_names:
            raise ValueError("model_names is empty")

        chosen_model = self._router_rng.choice(self.model_names)
        model_scores_info = {
            name: {
                "pred_score": None,
                "bonus": None,
                "total_ucb": None,
                "routing": "random",
                "chosen": (name == chosen_model),
            }
            for name in self.model_names
        }
        return chosen_model, model_scores_info


def train_with_random_controls(**kwargs):
    """复用 train_with_controls，只替换路由器为随机版。"""
    global GreedyNeuralUCBModel
    _orig_cls = GreedyNeuralUCBModel
    GreedyNeuralUCBModel = RandomRoutingNeuralUCBModel
    try:
        return train_with_controls(**kwargs)
    finally:
        GreedyNeuralUCBModel = _orig_cls


report_random = train_with_random_controls(
    dataset_path="../data/final_evaluation_dataset_500.json",
    train_size=50,
    use_previous_params=False,
    state_file="random_model_state_latest.pt",
    raw_record_file="execution_records_random_50.json",
    per_question_report_file="per_question_metrics_random_50.json",
)

report_random["summary"]

Torch version: 2.10.0+cu128
CUDA available: True
CUDA runtime: 12.8
GPU count: 2
Current GPU: Quadro RTX 8000
CUDA is ready for training/inference.
Loaded .env: /home/guisy/MyExperiment/LLM_DAG_Routing_02/.env
✅ deepseek/deepseek-v3.2 -> OK
✅ deepseek/deepseek-v3.2 -> OK
Loaded 7 models from model_config.py
['qwen/qwen-2.5-7b-instruct', 'meta-llama/llama-3.1-8b-instruct', 'mistralai/mistral-small-3.2-24b-instruct', 'google/gemma-3-27b-it', 'meta-llama/llama-3.3-70b-instruct', 'qwen/qwen3.5-flash-02-23', 'mistralai/mistral-nemo']
Loading embedding model: BAAI/bge-small-en-v1.5...


'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name BAAI/bge-small-en-v1.5. Creating a new one with mean pooling.
'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


⚠️ extractor 初始化失败，已跳过: RuntimeError: Cannot send a request, as the client has been closed.


'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/./modules.json


✅ 成功加载数据集: ../data/final_evaluation_dataset_500.json
⚠️ extractor 不存在，正在尝试初始化...
Loading embedding model: BAAI/bge-small-en-v1.5...


Retrying in 1s [Retry 1/5].
No sentence-transformers model found with name BAAI/bge-small-en-v1.5. Creating a new one with mean pooling.
'[Errno 104] Connection reset by peer' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.